# AxelBot Alpha Training (Colab)

Train a first-pass directional model from `feature_dataset.csv` exported by:

```bash
python scripts/export_feature_dataset.py --input "logs/paper-live-*.jsonl" --output "data/feature_dataset.csv" --horizon-events 3 --threshold-bps 2.0
```

In [ ]:
!pip -q install pandas scikit-learn xgboost joblib

In [ ]:
from pathlib import Path

dataset_path = None
candidates = [
    Path('data/feature_dataset.csv'),
    Path('/content/feature_dataset.csv'),
]

for p in candidates:
    if p.exists():
        dataset_path = p
        break

if dataset_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No file uploaded')
        dataset_path = Path(next(iter(uploaded.keys())))
    except Exception as e:
        raise RuntimeError(
            'Could not find dataset. Place data/feature_dataset.csv or upload manually.'
        ) from e

print(f'Using dataset: {dataset_path}')

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import joblib

df = pd.read_csv(dataset_path)
print('rows:', len(df))
display(df.head(3))

In [ ]:
# Optional: drop neutral labels for a cleaner binary setup.
df = df[df['label'] != 0].copy()
print('rows after non-neutral filter:', len(df))

# Time-aware split to reduce leakage.
if 'ts' in df.columns:
    df['ts'] = pd.to_datetime(df['ts'], errors='coerce')
    df = df.sort_values('ts').reset_index(drop=True)

feature_cols = [
    'mid',
    'micro_price',
    'fair_value',
    'spread_bps',
    'imbalance',
    'order_flow_signal',
    'alpha_bps',
    'inventory',
    'news_bias_bps',
    'quote_bid_price',
    'quote_ask_price',
    'quote_bid_size',
    'quote_ask_size',
    'market_bid',
    'market_ask',
]

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing feature columns: {missing}')

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Map {-1, +1} -> {0, 1}
y = (df['label'] > 0).astype(int)

split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
ret_test = df['fwd_ret_bps'].iloc[split:].values

print('train:', len(X_train), 'test:', len(X_test))

In [ ]:
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced')),
])
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
proba_lr = lr.predict_proba(X_test)[:, 1]

print('Logistic Regression')
print(classification_report(y_test, pred_lr, digits=4))

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=42,
)
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)
proba_xgb = xgb.predict_proba(X_test)[:, 1]

print('XGBoost')
print(classification_report(y_test, pred_xgb, digits=4))

In [ ]:
def pnl_proxy(prob, fwd_ret_bps, thr=0.55):
    # Long if prob > thr, short if prob < 1-thr, else flat.
    pos = np.where(prob > thr, 1.0, np.where(prob < (1.0 - thr), -1.0, 0.0))
    pnl = pos * fwd_ret_bps
    return {
        'active_ratio': float((pos != 0).mean()),
        'mean_bps': float(pnl.mean()),
        'sum_bps': float(pnl.sum()),
    }

print('LR proxy:', pnl_proxy(proba_lr, ret_test, thr=0.55))
print('XGB proxy:', pnl_proxy(proba_xgb, ret_test, thr=0.55))

In [ ]:
out_dir = Path('artifacts')
out_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(lr, out_dir / 'alpha_logreg.joblib')
joblib.dump(xgb, out_dir / 'alpha_xgb.joblib')

meta = {
    'feature_columns': feature_cols,
    'label_definition': 'sign of forward return beyond threshold, mapped to binary up/down',
    'threshold_note': 'training notebook drops neutral labels (label==0)',
}
with open(out_dir / 'alpha_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

print('Saved:', out_dir / 'alpha_logreg.joblib')
print('Saved:', out_dir / 'alpha_xgb.joblib')
print('Saved:', out_dir / 'alpha_meta.json')